# Modelo de accesibilidad (Nearest facility)

Cálculo de tiempos de caminata hacia el vecino más cercano mediante Multi-source Dijkstra.

## Configuración de entorno

Importamos librerías y dependencias.

In [ ]:
import geopandas as gpd
import osmnx as ox
import networkx as nx
from pathlib import Path

Configuración de OSMnx.

In [ ]:
ox.settings.use_cache = True
ox.settings.log_console = False

Configuración de rutas.

In [ ]:
PROCESSED_DIR = Path("../data/processed")
PROCESSED_FILE = PROCESSED_DIR / "lugares_datos_utiles_limpio.geojson"
GRAPH_FILE = PROCESSED_DIR / "grafo_rosario.graphml"

## Carga de datos

In [ ]:
gdf = gpd.read_file(PROCESSED_FILE)
grafo = ox.load_graphml(GRAPH_FILE)

In [ ]:
nodos, calles = ox.graph_to_gdfs(grafo)

## Snapping

In [ ]:
# Obtenemos los IDs de los nodos más cercanos a cada punto de interés
gdf['nodo'] = ox.distance.nearest_nodes(
    grafo,
    X=gdf.geometry.x,
    Y=gdf.geometry.y
)

In [ ]:
display(gdf[['name', 'categoria', 'nodo']].head())

## Multi-source Dijkstra

In [ ]:
categorias = gdf['categoria'].unique()

print("Iniciando cálculo de isocronas inversas (Dijkstra múltiple)...")

for categoria in categorias:
    print(f"Procesando: {categoria}...")
    
    # Agarramos todos los nodos donde hay un lugar de esa categoría
    nodos_origen = gdf[gdf['categoria'] == categoria]['nodo'].tolist()
    
    # Corremos el algoritmo: devuelve un diccionario {nodo_destino: tiempo_en_minutos}
    # Dijkstra inunda la red desde los orígenes y anota el camino más corto a cada nodo
    tiempos = nx.multi_source_dijkstra_path_length(grafo, nodos_origen, weight='walk_time')
    
    # Mapeamos ese diccionario a nuestro GeoDataFrame de esquinas
    # Si a un nodo no se puede llegar, le ponemos NaN
    nodos[f"time_{categoria.lower().replace(' ', '_')}"] = nodos.index.map(tiempos)

print("¡Cálculos finalizados!")

In [ ]:
# Filtramos buscando las columnas que acabamos de crear
cols_tiempos = [col for col in nodos.columns if col.startswith("time_")]

# Llenamos vacíos con 999 (inaccesible)
nodos[cols_tiempos] = nodos[cols_tiempos].fillna(999)

# Calculamos cuáles cumplen la meta de los 15 minutos
cobertura_15m = nodos[cols_tiempos] <= 15
nodos['score_15min'] = cobertura_15m.sum(axis=1)

print("\nDistribución del score de 15 Minutos en Rosario (Cantidad de esquinas):")
print(nodos['score_15min'].value_counts().sort_index(ascending=False))